In [1]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression

# 1) 사전학습 BERT 로드 - mBERT는 한국어를 포함한 다국어 지원 모델
MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert = AutoModel.from_pretrained(MODEL_NAME)
bert.eval()  # 학습 안 함 → 특징(임베딩) 추출기로만 사용

# 2) 임베딩 함수 - 문장을 입력 받고, 이 문장의 [CLS] 벡터를 산출
@torch.no_grad()
def cls_embedding(texts, max_length=64):
    enc = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
    outputs = bert(**enc)
    cls_vecs = outputs.last_hidden_state[:, 0, :] # 0번째 토큰이 [CLS]
    return cls_vecs.cpu().numpy()

# 3) 미니 학습 데이터 만들기(이진분류기 학습용)
X_train = [
    # 긍정(1)
    "정말 재미있고 감동적이었어요",
    "연기가 훌륭하고 볼거리가 풍부합니다",
    "음악과 연출이 인상적이고 몰입됐어요",
    "유머가 적절하고 전체적으로 만족스러웠어요",
    # 부정(0)
    "완전 지루하고 시간 아까웠어요",
    "스토리가 빈약하고 연기가 어색해요",
    "전개가 답답하고 몰입이 안 되네요",
    "볼만한 장면이 거의 없고 실망스러웠어요",
]
y_train = [1,1,1,1, 0,0,0,0]

# 4) 임베딩 추출(BERT는 고정) → 분류기 학습
Z_train = cls_embedding(X_train)
classifier = LogisticRegression(max_iter=1000)
classifier.fit(Z_train, y_train)

# 5) 예측 실습
print("\n모델 준비 완료! \n\n감성(긍정/부정) 예측을 테스트해보세요.")
print("예시) '배우들 연기가 인상적이고 몰입됐어요', '지루해서 중간에 나왔습니다'\n")

label_map = {0: "부정", 1: "긍정"}

while True:
    user_text = input("문장을 입력하세요(엔터만 누르면 종료): ")
    if user_text == "":
        print("종료합니다.")
        break
    # 임베딩 → 확률 예측
    z = cls_embedding([user_text])
    proba = classifier.predict_proba(z)[0]  # [P(0), P(1)]
    pred = int(np.argmax(proba))
    print(f"예측 결과: {label_map[pred]} (부정확률={proba[0]:.3f}, 긍정확률={proba[1]:.3f})\n")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



모델 준비 완료! 

감성(긍정/부정) 예측을 테스트해보세요.
예시) '배우들 연기가 인상적이고 몰입됐어요', '지루해서 중간에 나왔습니다'

문장을 입력하세요(엔터만 누르면 종료): 정말 재미있고 감동적이었어요
예측 결과: 긍정 (부정확률=0.253, 긍정확률=0.747)

문장을 입력하세요(엔터만 누르면 종료): 전개가 답답하고 몰입이 안 되네요
예측 결과: 부정 (부정확률=0.817, 긍정확률=0.183)

문장을 입력하세요(엔터만 누르면 종료): 음악과 영상은 인상적이다
예측 결과: 긍정 (부정확률=0.108, 긍정확률=0.892)

문장을 입력하세요(엔터만 누르면 종료): 
종료합니다.
